### Mugrade boilerplate

In [1]:
### Run this cell to install and import the homework tests
!pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part3_llm_training_tests.py

import mugrade
import os
from part3_llm_training_tests import *

def _mugrade_name(name):
    def rename(function):
        function.__name__ = name
        return function
    return rename

os.environ["MUGRADE_HW"] = "Part 3 - LLM Training"
os.environ["MUGRADE_KEY"] = "wl6RWtEQRAR7IlbJ9cs1" ### Your key here

  Cloning https://github.com/locuslab/mugrade.git to /tmp/pip-req-build-6t0ca7c_
  Running command git clone --filter=blob:none --quiet https://github.com/locuslab/mugrade.git /tmp/pip-req-build-6t0ca7c_
  Resolved https://github.com/locuslab/mugrade.git to commit 717e300a5c2ddc0c729746946f8dc9f0d1c0ecea
  Preparing metadata (setup.py) ... done
  Created wheel for mugrade: filename=mugrade-1.3-py3-none-any.whl size=4151 sha256=c43c1766b1169de9b43c31a1ca46e8f441e39930a80817c5674950aaf56711e8
  Stored in directory: /tmp/pip-ephem-wheel-cache-4z39cgzd/wheels/81/4d/29/7710726856b4a81e6438fa937bc2091df3fa0488ebcc452feb
Successfully built mugrade


--2026-09-10 01:02:34--  https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part3_llm_training_tests.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24254 (24K) [text/plain]
Saving to: ‘part3_llm_training_tests.py’

part3_llm_training_ 100%[===================>]  23.69K  --.-KB/s    in 0.002s  

2026-09-10 01:02:34 (11.6 MB/s) - ‘part3_llm_training_tests.py’ saved [24254/24254]



### BPE

Insert the BPE tokenizer you built in Part 1.

In [19]:
### BEGIN YOUR CODE
import json
from collections import Counter
import re
from tqdm.auto import tqdm
class BPE:
    #@mugrade.submit_tests
    def __init__(self, filename=None):
        ### BEGIN YOUR CODE
        if filename:
          with open(filename, "rt") as f:
            self.vocab, merge_keys, merge_values, self.special_tokens = json.load(f)
            self.special_tokens = {int(k): v for k, v in self.special_tokens.items()}
            self.merges = {tuple(k): v for k, v in zip(merge_keys, merge_values)}
        else:
          self.vocab = [chr(i) for i in range(256)]
          self.merges = {}
          self.special_tokens = {}
        ### END YOUR CODE

    @staticmethod
    #@mugrade.submit_tests
    def merge_pair(word, a, b, merged):
        ### BEGIN YOUR CODE
        i = 0
        while i<len(word)-1:
          if word[i] == a and word[i+1] == b:
            # merge
            word[i:i+2] = [merged]
          i+=1

        ### END YOUR CODE

    #@mugrade.submit_tests
    def replace_special_tokens(self, word):
        ### BEGIN YOUR CODE
        for k,v in self.special_tokens.items():
          word = word.replace(v, chr(k))
        return word
        ### END YOUR CODE


    #@mugrade.submit_tests
    def train(self, text, target_vocab_size):
        ### BEGIN YOUR CODE
        freqs = Counter(re.findall(r"\s\S+|\s+|\S+", text))
        freqs = {self.replace_special_tokens(k) :v for k, v in freqs.items() if v>1}
        words = [[ord(i) for i in word if ord(i) not in self.special_tokens] for word in freqs]
        word_count = freqs.values()
        for _ in range(target_vocab_size-len(self.vocab)):
          pair_count = Counter()
          for word, count in zip(words, word_count):
            # count pair
            for i in range(len(word)-1):
              pair_count[(word[i], word[i+1])] += count
          # a, b are ids
          a, b = pair_count.most_common(1)[0][0]
          new_id = len(self.vocab)
          for word in words:
            self.merge_pair(word, a, b, new_id)
          self.merges[(a,b)] = new_id
          self.vocab.append(self.vocab[a]+self.vocab[b])
        self.vocab += [""] * (max(self.special_tokens.keys()) - len(self.vocab) + 1)
        for k, v in self.special_tokens.items():
          self.vocab[k] = v


        ### END YOUR CODE

    #@mugrade.submit_tests
    def encode(self, text):
        ### BEGIN YOUR CODE
        tokens = []
        for m in re.finditer(r"\s\S+|\s+|\S+", text):
          word = [ord(i) for i in self.replace_special_tokens(m.group(0))]
          while True:
            merges = {}
            for pair in zip(word[:-1], word[1:]):
              if pair in self.merges:
                merges[pair] = self.merges[pair]
            if len(merges)==0:
              break
            a, b = min(merges, key=merges.get)
            self.merge_pair(word, a, b, merges[(a,b)])
          tokens+=word
        return tokens




        ### END YOUR CODE

    #@mugrade.submit_tests
    def decode(self, tokens):
        ### BEGIN YOUR CODE
        return "".join([self.vocab[t] for t in tokens])
        ### END YOUR CODE

    #@mugrade.submit_tests
    def save(self, fname):
        ### BEGIN YOUR CODE
        with open(fname, "wt") as f:
          json.dump([self.vocab, list(self.merges.keys()), list(self.merges.values()), self.special_tokens], f)
        ### END YOUR CODE
### END YOUR CODE

### LLM Architecture

Insert your LLM architecture from Part 2.

In [21]:
### BEGIN YOUR CODE
import torch
import math

#@mugrade.submit_tests
def embedding(x, weights, dtype):
    ### BEGIN YOUR CODE
    return weights[x].to(dtype=dtype)
    ### END YOUR CODE

#@mugrade.submit_tests
def linear(x, weights):
    ### BEGIN YOUR CODE
    return x @ weights.type_as(x)
    ### END YOUR CODE


#@mugrade.submit_tests
def silu(x):
    ### BEGIN YOUR CODE
    return x/(1+torch.exp(-x))
    ### END YOUR CODE

#@mugrade.submit_tests
def rms_norm(x):
    ### BEGIN YOUR CODE
    x_ = x.float()
    return (x_ / (x**2).mean(dim=-1, keepdim=True).sqrt() ).type_as(x)
    ### END YOUR CODE

#@mugrade.submit_tests
def softmax(x):
    ### BEGIN YOUR CODE
    x = (x - x.max(dim=-1, keepdim=True)[0]).exp()
    return x / x.sum(dim=-1, keepdim=True)
    ### END YOUR CODE

#@mugrade.submit_tests
def self_attn(q,k,v,mask):
    ### BEGIN YOUR CODE
    a = q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1]) + mask
    return softmax(a) @ v
    ### END YOUR CODE

class LLM:
    #@mugrade.submit_tests
    def rope(self, x):
        ### BEGIN YOUR CODE
        return self.buffers["rope1"] * x + self.buffers["rope2"] * x.reshape(-1,2).flip(-1).reshape_as(x)
        ### END YOUR CODE

    #@mugrade.submit_tests
    def multihead_attn(self, x, layer, mask):
      ### BEGIN YOUR CODE
      B, L, d = x.shape

      q = linear(x, self.params[f"wq_{layer}"])
      k = linear(x, self.params[f"wk_{layer}"])
      v = linear(x, self.params[f"wv_{layer}"])

      q = q.reshape(B, L, self.num_heads, self.head_dim).transpose(1,2)
      k = k.reshape(B, L, self.num_heads, self.head_dim).transpose(1,2)
      v = v.reshape(B, L, self.num_heads, self.head_dim).transpose(1,2)

      q = rms_norm(self.rope(q))
      k = rms_norm(self.rope(k))

      out = self_attn(q, k, v, mask)
      return linear(out.transpose(1,2).reshape_as(x), self.params[f"wp_{layer}"])
      ### END YOUR CODE

    #@mugrade.submit_tests
    def mlp(self, x, layer):
        ### BEGIN YOUR CODE
      return linear(silu(linear(x, self.params[f"w1_{layer}"])), self.params[f"w2_{layer}"])
        ### END YOUR CODE

    #@mugrade.submit_tests
    def transformer_block(self, x, layer, mask):
      ### BEGIN YOUR CODE
      x = x + self.multihead_attn(rms_norm(x), layer, mask)
      return x + self.mlp(rms_norm(x), layer)
      ### END YOUR CODE

    @mugrade.submit_tests
    def __init__(self, config):
      ### BEGIN YOUR CODE
      d = config["depth"] * config["aspect_ratio"]
      mlp_d = d * config["mlp_multiple"]
      self.num_layers = config["depth"]
      self.head_dim = config["head_dim"]
      self.num_heads = d // self.head_dim
      self.dtype = config["dtype"]

      self.params = {}
      scale, scale_mlp = math.sqrt(2/d), math.sqrt(2/mlp_d)
      self.params["output"] = torch.randn(d, config["vocab_size"]) * scale
      self.params["embedding"] = torch.randn(config["vocab_size"],d) * scale

      for i in range(self.num_layers):
          self.params[f"wq_{i}"] = torch.randn(d,d) * scale
          self.params[f"wk_{i}"] = torch.randn(d,d) * scale
          self.params[f"wv_{i}"] = torch.randn(d,d) * scale
          self.params[f"wp_{i}"] = torch.randn(d,d) * scale
          self.params[f"w1_{i}"] = torch.randn(d,mlp_d) * scale
          self.params[f"w2_{i}"] = torch.randn(mlp_d,d) * scale_mlp
      for k, v in self.params.items():
        self.params[k] = v.to(dtype=self.dtype)

      self.buffers = {}
      mask = torch.full((config["seq_len"], config["seq_len"]), -float("inf"))
      self.buffers["mask"] = torch.triu(mask, diagonal=1).to(dtype=self.dtype)
      freqs = config["rope_theta"] ** (-2 * torch.arange(self.head_dim//2) / self.head_dim)
      theta = torch.outer(torch.arange(config["seq_len"]), freqs)
      cos, sin = torch.cos(theta), torch.sin(theta)
      self.buffers["rope1"] = torch.stack([cos, cos], dim=-1).reshape(config["seq_len"], self.head_dim)
      self.buffers["rope2"] = torch.stack([-sin, sin], dim=-1).reshape(config["seq_len"], self.head_dim)

      ### END YOUR CODE

    #@mugrade.submit_tests
    def __call__(self, tokens):
      ### BEGIN YOUR CODE
      x = rms_norm(embedding(tokens, self.params["embedding"], self.dtype))
      for i in range(self.num_layers):
          x = self.transformer_block(x, i, self.buffers["mask"])
      return linear(rms_norm(x), self.params["output"])
      ### END YOUR CODE

    #@mugrade.submit_tests
    def save(self, filename):
        ### BEGIN YOUR CODE
        torch.save(self.params, filename)
        ### END YOUR CODE

    #@mugrade.submit_tests
    def load(self, filename):
        ### BEGIN YOUR CODE
        self.params = torch.load(filename, weights_only=True)
        ### END YOUR CODE

### END YOUR CODE

Mugrade: Submitting tests for function __init__():
ERROR: No submission function found


### Downloading necessary files

These next lines will donwload the necessary tokenizer and pre-tokenized data files, which you can build in Part 1 (but which require a fairly substantial machine to run as-is).

In [2]:
from huggingface_hub import hf_hub_download
import os

repo = "zkolter/llm_speedrun"
filenames = ["fineweb-edu-10BT.shuffle.bin", "smoltalk.shuffle.bin", "tokenizer_50M.bpe"]

for filename in filenames:
    if not os.path.exists(filename):
        hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset", local_dir=".")

fineweb-edu-10BT.shuffle.bin: reconstructing file:   0%|          |  0.00B / 18.9GB            

fineweb-edu-10BT.shuffle.bin: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

You can use the following config file for training.  This is intended to load a batch size that will fit comfortably on a GPU with 80GB of memory, you can adjust as needed.

In [3]:
%%writefile config.d12.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "mlp_multiple": 4,
    "head_dim": 128,
    "dtype": "bfloat16",
    "vocab_size": 32768,
    "batch_size": 16,
    "seq_len": 2048,
    "rope_theta": 10000,
    "tokenizer": "tokenizer_50M.bpe",
    "token_multiple": 20,
    "lr": 7e-4,
    "weight_decay": 0.025,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "num_gpus": 1
}

Writing config.d12.json


### LLM Training

In [24]:
import wandb
import time
from array import array
import os
import torch
import math


#@mugrade.submit_tests
def cross_entropy_loss(logits, y):
    ### BEGIN YOUR CODE
    logits = logits - logits.max(dim=-1, keepdim=True)[0]
    loss = -logits.take_along_dim(y.unsqueeze(-1), dim=-1)
    loss += logits.exp().sum(dim=-1, keepdim=True).log()
    return loss.mean()
    ### END YOUR CODE

class Adam:
    #@mugrade.submit_tests
    @_mugrade_name("Adam_init")
    def __init__(self, params, lr_schedule, betas = (0.9, 0.95), eps=1e-5, weight_decay=0.0):
        ### BEGIN YOUR CODE
        self.schedule = lr_schedule
        self.betas = betas
        self.params = params
        self.t = 1
        self.weight_decay = weight_decay
        self.u = {k: torch.zeros_like(v) for k,v in params.items()}
        self.v = {k: torch.zeros_like(v) for k,v in params.items()}
        self.eps = eps
        for p in self.params.values():
            p.requires_grad_()
        ### END YOUR CODE

    #@mugrade.submit_tests
    def step(self):
        ### BEGIN YOUR CODE
        with torch.no_grad():
          for k,p in self.params.items():
              self.u[k] = self.betas[0]*self.u[k] + (1-self.betas[0])*p.grad
              self.v[k] = self.betas[1]*self.v[k] + (1-self.betas[1])*p.grad**2
              p.grad.zero_()

              u_hat = self.u[k] / (1 - self.betas[0]**self.t)
              v_hat = self.v[k] / (1 - self.betas[1]**self.t)

              p *= (1 - self.schedule.get_lr(self.t) * self.weight_decay)
              p -= self.schedule.get_lr(self.t) * u_hat / (v_hat.sqrt() + self.eps)
        self.t += 1
        ### END YOUR CODE

class LRSchedule:
    #@mugrade.submit_tests
    @_mugrade_name("LRSchedule_init")
    def __init__(self, total_steps, lr=1e-3, warmup_steps=50, decay_ratio=0.4, min_frac=0.1):
        ### BEGIN YOUR CODE
        self.total_steps = total_steps
        self.lr = lr
        self.warmup_steps = warmup_steps
        self.decay_steps = round(decay_ratio * total_steps)
        self.min_frac = min_frac
        ### END YOUR CODE

    #@mugrade.submit_tests
    def get_lr(self, step):
        ### BEGIN YOUR CODE
        if step < self.warmup_steps:
            p = step / self.warmup_steps
            return (1-p)*self.min_frac*self.lr + p*self.lr
        elif step > self.total_steps - self.decay_steps:
            p = (step - (self.total_steps - self.decay_steps)) / self.decay_steps
            return (1-p)*self.lr + p*self.lr*self.min_frac
        else:
            return self.lr
        ### END YOUR CODE


@mugrade.submit_tests
def train_llm(config, log=False):
    ### BEGIN YOUR CODE
    llm = LLM(config)
    tokenizer = BPE(config["tokenizer"])
    for k in llm.params: llm.params[k] = llm.params[k].cuda()
    for k in llm.buffers: llm.buffers[k] = llm.buffers[k].cuda()

    config["total_params"] = sum(p.numel() for p in llm.params.values())
    config["total_tokens"] = config["total_params"] * config["token_multiple"]
    config["total_steps"] = config["total_tokens"] // (config["batch_size"] * config["seq_len"])
    schedule = LRSchedule(config["total_steps"], config["lr"])
    opt = Adam(llm.params, schedule, weight_decay=config["weight_decay"])
    print("CONFIG TYPES:")
    for k, v in config.items():
        print(k, type(v))
    if log:
      run = wandb.init(project="llm_speedrun", config=config)

    batch_items = []
    for k,p in config["data_mix"].items():
        batch_items += [(k, i*(config["seq_len"]+1)*2) for i in range(round(p*config["batch_size"]))]
    file_offsets = {k:0 for k in config["data_mix"]}
    n_tok = 0

    while opt.t < config["total_steps"]:
        start_time = time.perf_counter()
        tokens = []
        for filename, offset in batch_items:
            with open(filename, "rb") as f:
                f.seek(file_offsets[filename] + offset)
                tokens += array("H", f.read((config["seq_len"]+1)*2)).tolist()

        for k,p in config["data_mix"].items():
            read_size = round(p*config["batch_size"]) * (config["seq_len"]+1) * 2
            file_offsets[k] += read_size
            if file_offsets[k] + read_size >= os.path.getsize(k):
              file_offsets[k] = 0

        tokens = torch.tensor(tokens).reshape(config["batch_size"], config["seq_len"]+1)
        text = tokenizer.decode(tokens[:,1:].flatten())

        # run and take a gradient
        tokens = tokens.cuda()
        logits = llm(tokens[:,:-1]).float()
        loss = cross_entropy_loss(logits, tokens[:,1:])
        loss.backward()
        opt.step()

        bpb = loss.item() * config["batch_size"] * config["seq_len"] / (len(text) * math.log(2))
        n_tok += config["batch_size"] * config["seq_len"]
        tok_per_sec = config["batch_size"] * config["seq_len"] / (time.perf_counter() - start_time)

        if log:
          run.log({
              "step": opt.t,
              "tokens": n_tok,
              "loss": loss.item(),
              "bpb": bpb,
              "lr": opt.schedule.get_lr(opt.t),
              "tok_per_sec": tok_per_sec})
    print("LLM PARAM TYPE:", type(next(iter(llm.params.values()))))
    return llm

    ### END YOUR CODE

Mugrade: Submitting tests for function train_llm():
CONFIG TYPES:
tokenizer <class 'str'>
token_multiple <class 'int'>
batch_size <class 'int'>
seq_len <class 'int'>
lr <class 'float'>
weight_decay <class 'float'>
data_mix <class 'dict'>
total_params <class 'int'>
total_tokens <class 'int'>
total_steps <class 'int'>
Grader test 1 passed


TypeError: Object of type Tensor is not JSON serializable

If your code passes all the tests, you can (optionally) uncomment this block to train the d12 model, here done on a single GPU.

In [ ]:
with open("config.d12.class.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]

llm = train_llm(config, log=True)

### Distributed LLM Training

Implement a distributed version of the training code, starting from the version above.  Note that there was one error in the class version, that `config["batch_size"]` should be replaced by `local_batch_size` in the line that computes bpb for logging.

In [ ]:
# @mugrade.local_tests
def train_llm_distributed(rank, nccl_uid, config, log=False):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

If you're able to access a machine with 8 GPUs, then the following code will launch the distributed training run with a different config that uses a larger batch size.

In [ ]:
%%writefile config.d12.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "mlp_multiple": 4,
    "head_dim": 128,
    "dtype": "bfloat16",
    "vocab_size": 32768,
    "batch_size": 128,
    "seq_len": 2048,
    "rope_theta": 10000,
    "tokenizer": "tokenizer_50M.bpe",
    "token_multiple": 20,
    "lr": 1e-3,
    "weight_decay": 0.1,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "num_gpus": 8
}

In [ ]:
from joblib import Parallel, delayed
os.environ["NCCL_NVLS_ENABLE"] = "0"  # shouldn't be needed if your system isn't messed up like mine

with open("config.d12.class.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]
nccl_uid = torch.cuda.nccl.unique_id()

Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(train_llm_distributed)(i, nccl_uid, config, log=True) for i in range(config["num_gpus"])
)
